# Customer Complaint Category Classification

**Classification Project 24 of 100** — Classify customer complaints into product categories.

EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.

## 2. Project Overview

Classify customer complaint narratives into product categories (Credit reporting, Debt collection, Mortgage, etc.) using **TF-IDF + traditional ML**. This is a text classification task with tabular ML methods — a strong baseline approach before deep learning.

We use the CFPB Consumer Complaints dataset. The dataset is large, so we sample and use the top categories.

## 3. Learning Objectives

1. Text classification with TF-IDF vectorization
2. Multi-class with many categories (top 5 selected)
3. Text preprocessing (lowercasing, stopwords)
4. LazyPredict and FLAML on text features
5. Confusion patterns between similar products
6. NLP basics without deep learning

## 4. Problem Statement

> Given a consumer complaint text, predict the product category it belongs to.

Multi-class text classification. Macro-F1 is the primary metric.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Customer service | Auto-routing complaints |
| Regulators | Trend monitoring |
| Companies | Response time reduction |
| ML learners | Text classification basics |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | data.gov / Kaggle |
| Kaggle slug | selener/consumer-complaint-database |
| Rows | ~1M+ (sampled to 20k) |
| Text column | Consumer complaint narrative |
| Target | Product (top 5 categories) |
| Balance | Imbalanced |

## 7. Dataset Source and License Notes

- CFPB Consumer Complaint Database
- Kaggle: selener/consumer-complaint-database
- License: Public domain (US Government data)
- Contains PII-redacted complaint narratives.

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, ConfusionMatrixDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "selener/consumer-complaint-database"
TEXT_COL = "Consumer complaint narrative"
TARGET = "Product"
MAX_ROWS = 20000; TOP_K_CLASSES = 5
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 90
TFIDF_MAX_FEATURES = 5000
print(f"Target: {TARGET}, Max rows: {MAX_ROWS}")

## 11. Dataset Download or Loading

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

In [ ]:
# Keep only rows with complaint narrative
df = df_raw.dropna(subset=[TEXT_COL, TARGET]).copy()
print(f"After dropping missing text/target: {len(df)}")
# Keep top K categories
top_cats = df[TARGET].value_counts().head(TOP_K_CLASSES).index.tolist()
df = df[df[TARGET].isin(top_cats)].reset_index(drop=True)
print(f"After keeping top {TOP_K_CLASSES} categories: {len(df)}")
# Sample
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Sampled to: {len(df)}")
print(f"\nCategories:\n{df[TARGET].value_counts()}")

## 13. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
df[TARGET].value_counts().plot.barh(ax=ax, color=sns.color_palette("Set2"))
ax.set_title("Complaint Categories"); ax.set_xlabel("Count")
plt.tight_layout(); plt.show()

In [ ]:
# Text length analysis
df["text_len"] = df[TEXT_COL].str.len()
fig, ax = plt.subplots(figsize=(8,4))
for cat in top_cats[:5]:
    ax.hist(df[df[TARGET]==cat]["text_len"].clip(upper=3000), bins=50, alpha=0.4, label=cat[:20])
ax.set_title("Complaint Text Length by Category"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
print(f"Avg text length: {df['text_len'].mean():.0f} chars")
print(f"Median: {df['text_len'].median():.0f}")
print(f"Max: {df['text_len'].max()}")

## 14. Target Analysis

Imbalanced across categories. Credit reporting complaints dominate.

In [ ]:
print(df[TARGET].value_counts(normalize=True))

## 15. Train / Validation / Test Split

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
X_text = df[TEXT_COL].values
X_tmp, X_test_text, y_tmp, y_test = train_test_split(X_text, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train_text, X_val_text, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {len(X_train_text)}  Val: {len(X_val_text)}  Test: {len(X_test_text)}")
print(f"Classes: {le.classes_}")

## 16. Preprocessing Strategy

TF-IDF vectorization with max 5000 features, English stopwords, sublinear TF.

In [ ]:
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, stop_words="english",
                        sublinear_tf=True, ngram_range=(1,2), min_df=3)
X_train = tfidf.fit_transform(X_train_text)
X_val = tfidf.transform(X_val_text)
X_test = tfidf.transform(X_test_text)
print(f"TF-IDF shape: {X_train.shape}")

## 17. Feature Engineering

TF-IDF already serves as feature engineering. Bigrams included for phrase capture. No additional hand-crafted features needed for this baseline approach.

In [ ]:
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Top features: {tfidf.get_feature_names_out()[:20].tolist()}")

## 18. Baseline Model

In [ ]:
results = {}
for name, clf in [("Dummy",DummyClassifier(strategy="most_frequent",random_state=SEED)),
                  ("MultinomialNB",MultinomialNB()),
                  ("LogReg",LogisticRegression(max_iter=1000,random_state=SEED))]:
    clf.fit(X_train, y_train); yp=clf.predict(X_val)
    r={"Accuracy":accuracy_score(y_val,yp),"Macro-F1":f1_score(y_val,yp,average="macro")}
    results[name]=r; print(f"{name}: {r}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_d = X_train.toarray(); X_val_d = X_val.toarray()
lazy=LazyClassifier(verbose=0,ignore_warnings=True,random_state=SEED)
models_lp,_=lazy.fit(X_train_d,X_val_d,y_train,y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
automl=AutoML()
automl.fit(X_train_d,y_train,task="classification",metric="macro_f1",time_budget=FLAML_BUDGET,seed=SEED,verbose=0)
print(f"Best: {automl.best_estimator}")
yp_fl=automl.predict(X_val_d)
results["FLAML"]={"Accuracy":accuracy_score(y_val,yp_fl),"Macro-F1":f1_score(y_val,yp_fl,average="macro")}
print(results["FLAML"])

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm=True
except ImportError: has_lgbm=False
try:
    from xgboost import XGBClassifier; has_xgb=True
except ImportError: has_xgb=False
top3={}
top3["LogReg_tuned"]=LogisticRegression(C=5.0, max_iter=2000, random_state=SEED)
if has_lgbm: top3["LightGBM"]=LGBMClassifier(n_estimators=300,learning_rate=0.1,max_depth=6,random_state=SEED,verbose=-1,n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"]=ExtraTreesClassifier(n_estimators=300,random_state=SEED,n_jobs=-1)
if has_xgb: top3["XGBoost"]=XGBClassifier(n_estimators=300,learning_rate=0.1,max_depth=6,random_state=SEED,verbosity=0,n_jobs=-1,eval_metric="mlogloss")
else:
    top3["RF"]=RandomForestClassifier(n_estimators=300,random_state=SEED,n_jobs=-1)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_test_d = X_test.toarray()
labels = le.classes_.tolist()
# Truncate labels for display
short_labels = [l[:25] for l in labels]
final={}
for name,mdl in top3.items():
    mdl.fit(X_train_d,y_train); yp=mdl.predict(X_test_d)
    final[name]={"Accuracy":accuracy_score(y_test,yp),"Macro-F1":f1_score(y_test,yp,average="macro")}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=short_labels))
pd.DataFrame(final).T

In [ ]:
fig,axes=plt.subplots(1,len(top3),figsize=(6*len(top3),5))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes,top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_test_d),ax=ax,cmap="Blues",display_labels=short_labels)
    ax.set_title(n); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
bm=list(top3.values())[0]; ypb=bm.predict(X_test_d)
misclass = (y_test != ypb).sum()
print(f"Misclassified: {misclass}/{len(y_test)} ({misclass/len(y_test):.1%})")
from collections import Counter
confused = Counter(zip(le.inverse_transform(y_test[y_test!=ypb]),
                       le.inverse_transform(ypb[y_test!=ypb])))
print("\nMost confused pairs:")
for pair, cnt in confused.most_common(5):
    print(f"  {pair[0][:30]} -> {pair[1][:30]}: {cnt}")

## 24. Interpretation and Business Insight

TF-IDF + LogReg is a strong baseline for complaint routing. Domain-specific terms (mortgage, credit, debt) provide clear signals. Confusion typically occurs between related financial products.

## 25. Limitations

1. TF-IDF loses word order and context
2. Only top 5 categories used
3. PII redaction creates XXXX tokens
4. No deep learning (BERT would improve)
5. Sampled subset only

## 26. How to Improve This Project

1. Fine-tune BERT/DistilBERT
2. Use all product categories
3. Add metadata features (state, company)
4. Hierarchical classification
5. Active learning for edge cases

## 27. Production Considerations

- Real-time complaint routing
- Confidence threshold for human review
- Category taxonomy changes over time
- Multi-label extension (complaints spanning products)

## 28. Common Mistakes

1. Not removing XXXX redaction tokens
2. Using accuracy with imbalanced classes
3. Not using sublinear TF
4. Too many TF-IDF features (overfitting)
5. Ignoring similar category confusion

## 29. Mini Challenge / Exercises

1. Only unigrams vs bigrams — difference?
2. Add text length as extra feature
3. Use all categories instead of top 5
4. Remove XXXX tokens — improvement?
5. Character n-grams vs word n-grams

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Multi-class text classification |
| Dataset | CFPB complaints, 20k sample |
| Difficulty | Medium |
| Key insight | Domain terms separate categories |

TF-IDF + traditional ML as strong NLP baseline, full pipeline.